# Kaggle Submission Workflow

Notebook for producing and checking submission files from the repository scripts. Long-running training is disabled by default; flip the relevant `RUN_*` flag when you want the notebook to execute a command.


## Setup


In [ ]:
from pathlib import Path
import subprocess
import sys

import pyarrow.parquet as pq

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent

TRAIN_PATH = ROOT / "src" / "data" / "train.parquet"
TEST_PATH = ROOT / "src" / "data" / "test.parquet"
PREDICTIONS_DIR = ROOT / "outputs" / "predictions"
STACK_DIR = PREDICTIONS_DIR / "2-level_submission"
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)
STACK_DIR.mkdir(parents=True, exist_ok=True)

print(f"root={ROOT}")
print(f"python={sys.executable}")
print(f"train={TRAIN_PATH}")
print(f"test={TEST_PATH}")


## Confirm Input Files

This reads metadata only.


In [ ]:
for path in [TRAIN_PATH, TEST_PATH]:
    pf = pq.ParquetFile(path)
    print(f"{path.name}: {pf.metadata.num_rows:,} rows, {pf.metadata.num_columns:,} columns")


## Helper To Run Commands


In [ ]:
def run_command(args, *, run=False):
    command_text = " ".join(str(x) for x in args)
    print(command_text)
    if run:
        subprocess.run([str(x) for x in args], cwd=ROOT, check=True)


## Option A: Main 2-Level Stack

This is the primary submission path. It can take a long time depending on hardware.


In [ ]:
RUN_TWO_LEVEL_STACK = False

run_command(
    [
        sys.executable,
        ROOT / "scripts" / "run_two_level_submission.py",
        "--train-path", TRAIN_PATH,
        "--test-path", TEST_PATH,
        "--output-dir", STACK_DIR,
    ],
    run=RUN_TWO_LEVEL_STACK,
)


## Option B: CPU-Only 2-Level Stack

Use this if LightGBM or XGBoost GPU support is unavailable.


In [ ]:
RUN_TWO_LEVEL_STACK_CPU = False

run_command(
    [
        sys.executable,
        ROOT / "scripts" / "run_two_level_submission.py",
        "--train-path", TRAIN_PATH,
        "--test-path", TEST_PATH,
        "--output-dir", STACK_DIR,
        "--no-use-gpu",
    ],
    run=RUN_TWO_LEVEL_STACK_CPU,
)


## Option C: Baseline CV Then Submission

First run validation. Keep `--skip-final-fit` for validation only, or remove it to also train final models.


In [ ]:
RUN_CV_VALIDATION = False

run_command(
    [
        sys.executable,
        ROOT / "scripts" / "run_cv.py",
        "--train-path", TRAIN_PATH,
        "--test-path", TEST_PATH,
        "--skip-final-fit",
    ],
    run=RUN_CV_VALIDATION,
)


In [ ]:
RUN_BASELINE_SUBMISSION = False
validation_summary = PREDICTIONS_DIR / "validation_summary_full.json"

run_command(
    [
        sys.executable,
        ROOT / "scripts" / "make_submission.py",
        "--train-path", TRAIN_PATH,
        "--test-path", TEST_PATH,
        "--validation-summary", validation_summary,
    ],
    run=RUN_BASELINE_SUBMISSION,
)


## Check Submission File

Run this after generating a submission. It checks row count and basic column structure.


In [ ]:
import pandas as pd

submission_path = STACK_DIR / "submission.csv"
if not submission_path.exists():
    submission_path = PREDICTIONS_DIR / "submission.csv"

print(f"candidate submission: {submission_path}")
if submission_path.exists():
    submission = pd.read_csv(submission_path)
    print(submission.shape)
    display(submission.head())
    print(submission.dtypes)
else:
    print("No submission file found yet. Run one of the commands above first.")


## Expected Output Locations

- Main stack: `outputs/predictions/2-level_submission/submission.csv`
- Baseline submission: `outputs/predictions/submission.csv`
- CV summary: `outputs/predictions/validation_summary_full.json`
